# Exoplanet Pipeline Kaggle Real-Data Run

This notebook runs the pipeline on **real public TESS/TOI data by default**. It downloads public TOI metadata from NASA Exoplanet Archive, downloads real TESS light-curve FITS from MAST, extracts candidate features, trains the tabular AI classifier, runs the science batch pipeline on the downloaded FITS files, and generates final submission assets.

Synthetic data is not used unless you explicitly enable `RUN_SYNTHETIC_SMOKE=True` as an environment check.


## 1. System Diagnostics

This cell records the Kaggle runtime, GPU visibility, memory, and internet access. MAST/NASA downloads require internet.


In [ ]:
%%time
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request

import pandas as pd

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

print('Python executable:', sys.executable)
print('Python version:', sys.version)
print('Current directory:', Path.cwd())
print('\nKaggle input datasets:')
for path in sorted(INPUT.glob('*')):
    print(' -', path)

try:
    import psutil
    TOTAL_RAM_GB = psutil.virtual_memory().total / (1024 ** 3)
except Exception:
    TOTAL_RAM_GB = 16.0
print(f'\nTotal RAM: {TOTAL_RAM_GB:.2f} GB')

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    print('PyTorch:', torch.__version__)
    print('CUDA available:', GPU_AVAILABLE)
    if GPU_AVAILABLE:
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    GPU_AVAILABLE = False
    print('PyTorch/CUDA check failed:', repr(exc))

INTERNET_AVAILABLE = False
try:
    urllib.request.urlopen('https://exoplanetarchive.ipac.caltech.edu/', timeout=10)
    INTERNET_AVAILABLE = True
except Exception as exc:
    print('Internet check failed:', repr(exc))
print('Internet available:', INTERNET_AVAILABLE)


## 2. Configuration

The default mode is `public_toi`, which uses real public data. Switch to `curated` only if you have an organizer-provided label CSV attached under `/kaggle/input`.


In [ ]:
# Repository source. Keep this on main unless you are testing a branch.
REPO_URL = 'https://github.com/im-rajnandan/exoplanet-pipeline-upgraded.git'
REPO_BRANCH = 'main'
REPO_DIR = WORK / 'exoplanet-pipeline'

# If internet is disabled, attach the repo as a Kaggle dataset at this path.
REPO_DATASET_DIR = Path('/kaggle/input/exoplanet-pipeline-upgraded')

# Real-data mode.
# public_toi: download real public TOI metadata and real TESS FITS from MAST.
# curated: use a user/organizer-provided label file.
REAL_DATA_MODE = 'public_toi'

# Public TOI metadata size. This is metadata only, not FITS downloads.
PUBLIC_TOI_TOP = 2000
PUBLIC_DATA_DIR = WORK / 'data' / 'public'
PUBLIC_RAW_TOI_CSV = PUBLIC_DATA_DIR / 'metadata' / 'tess-toi_raw_metadata.csv'
PUBLIC_NORMALIZED_TOI_CSV = PUBLIC_DATA_DIR / 'metadata' / 'tess-toi_normalized_metadata.csv'
PUBLIC_TRAINING_CSV = PUBLIC_DATA_DIR / 'metadata' / 'tess-toi_training_subset.csv'

# Curated mode path. Used only when REAL_DATA_MODE == 'curated'.
CURATED_INPUT_CSV = Path('/kaggle/input/curated/curated_labels.csv')

# Light-curve storage. If LIGHTCURVE_DIR exists, it is used as a read-only cache.
# Missing files are downloaded to LIGHTCURVE_DOWNLOAD_DIR when USE_MAST_DOWNLOADS=True.
LIGHTCURVE_DIR = Path('/kaggle/input/tess-lightcurves')
LIGHTCURVE_DOWNLOAD_DIR = WORK / 'data' / 'lightcurves'
USE_MAST_DOWNLOADS = True

# Run sizes. Start with 200 real labeled rows; increase after this works.
MAX_LABEL_ROWS = 200
N_PERIODS_FEATURES = 3000
N_PERIODS_SCIENCE = 4000
N_BOOTSTRAP_FEATURES = 80
TIMEOUT_SECONDS = 300
NO_VARIANTS = True
FEATURE_CHECKPOINT_EVERY = 5

# Worker tuning. Feature building is mostly I/O + FITS work; too many workers can upset MAST or RAM.
if TOTAL_RAM_GB >= 28:
    N_WORKERS_FEATURES = 4
    N_WORKERS_SCIENCE = 2
elif TOTAL_RAM_GB >= 14:
    N_WORKERS_FEATURES = 2
    N_WORKERS_SCIENCE = 1
else:
    N_WORKERS_FEATURES = 1
    N_WORKERS_SCIENCE = 1

# Optional. Synthetic smoke is only an environment test and is off by default.
RUN_SYNTHETIC_SMOKE = False

# Optional CNN path. Leave off for the first real tabular run.
RUN_OPTIONAL_CNN = False
CNN_METADATA_TOP = 2000
CNN_MAX_ROWS = 2000
CNN_EPOCHS = 20
CNN_BATCH_SIZE = 32

# Optional validation path. Set to a final candidate catalog with labels already joined in.
LABELED_SCIENCE_CATALOG = None

# Outputs.
OUT_LABELS = WORK / 'outputs_labeled_candidates'
OUT_AI = WORK / 'outputs_part6_ai'
OUT_CNN = WORK / 'outputs_cnn'
OUT_SCIENCE = WORK / 'outputs_science'
OUT_VALIDATION = WORK / 'outputs_validation'

print('Mode:', REAL_DATA_MODE)
print('MAX_LABEL_ROWS:', MAX_LABEL_ROWS)
print('Feature workers:', N_WORKERS_FEATURES)
print('Science workers:', N_WORKERS_SCIENCE)
print('MAST downloads:', USE_MAST_DOWNLOADS)


## 3. Helpers

`run_command` streams output live, including carriage-return progress bars, so long MAST/FITS jobs show progress instead of appearing frozen.


In [ ]:
def run_command(cmd, cwd=None, check=True):
    print('\n$ ' + ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    chunks = []
    assert process.stdout is not None
    while True:
        chunk = process.stdout.read(4096)
        if chunk:
            text = chunk.decode('utf-8', errors='replace')
            print(text, end='', flush=True)
            chunks.append(text)
            continue
        if process.poll() is not None:
            break
    returncode = process.wait()
    output = ''.join(chunks)
    if check and returncode != 0:
        raise RuntimeError(f'Command failed with exit code {returncode}: {cmd}')
    return subprocess.CompletedProcess(cmd, returncode, output, None)


def count_fits(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for _ in path.rglob('*.fits')) + sum(1 for _ in path.rglob('*.fits.gz')) + sum(1 for _ in path.rglob('*.fits.fz'))


def assert_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{label} not found: {path}')
    print(f'{label}: {path}')

print('Helpers ready.')


## 4. Set Up Repository And Dependencies

This always pulls the latest `main` if the repo already exists. That prevents stale notebooks/scripts from being reused accidentally.


In [ ]:
%%time
if REPO_DIR.exists():
    print('Repository already exists:', REPO_DIR)
else:
    if REPO_DATASET_DIR.exists():
        print('Copying repository dataset:', REPO_DATASET_DIR)
        shutil.copytree(REPO_DATASET_DIR, REPO_DIR)
    elif INTERNET_AVAILABLE:
        run_command(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], cwd=WORK)
    else:
        raise RuntimeError('No repository found. Enable internet or attach repo dataset at /kaggle/input/exoplanet-pipeline-upgraded.')

if (REPO_DIR / '.git').exists() and INTERNET_AVAILABLE:
    run_command(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run_command(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    run_command(['git', 'pull', 'origin', REPO_BRANCH], cwd=REPO_DIR)

run_command(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

run_command([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run_command([sys.executable, '-m', 'pip', 'install', '-e', '.[mast]'], cwd=REPO_DIR)

# Kaggle GPU runtimes usually already include torch. Install only if optional CNN is enabled and torch is missing.
if RUN_OPTIONAL_CNN:
    run_command([sys.executable, '-m', 'pip', 'install', '-e', '.[deep]'], cwd=REPO_DIR)


## 5. Download Real Metadata And Build The Label Table

For the default public run, this downloads real TOI metadata and creates a balanced labeled subset. The actual light-curve FITS files are downloaded in the next step, because they require TIC IDs from this table.


In [ ]:
%%time
if REAL_DATA_MODE == 'public_toi':
    if not INTERNET_AVAILABLE and not PUBLIC_RAW_TOI_CSV.exists():
        raise RuntimeError('Public TOI mode needs internet for metadata unless the metadata CSV already exists.')

    if not PUBLIC_RAW_TOI_CSV.exists():
        run_command([
            sys.executable, 'scripts/download_public_metadata.py',
            '--source', 'tess-toi',
            '--output-dir', str(PUBLIC_DATA_DIR),
            '--top', str(PUBLIC_TOI_TOP),
        ], cwd=REPO_DIR)
    else:
        print('Public TOI raw metadata already exists:', PUBLIC_RAW_TOI_CSV)

    assert_exists(PUBLIC_NORMALIZED_TOI_CSV, 'normalized TOI metadata')
    meta = pd.read_csv(PUBLIC_NORMALIZED_TOI_CSV)
    print('Normalized metadata shape:', meta.shape)

    required_cols = ['tic_id', 'binary_label', 'period_days', 'epoch_time', 'duration_days']
    missing_cols = [c for c in required_cols if c not in meta.columns]
    if missing_cols:
        raise ValueError(f'Normalized metadata missing columns: {missing_cols}')

    usable = meta.dropna(subset=required_cols).copy()
    usable = usable[usable['binary_label'].isin(['planet_like', 'false_positive_or_other'])].copy()
    print('Usable labeled metadata rows:', usable.shape)
    print(usable['binary_label'].value_counts(dropna=False))

    if MAX_LABEL_ROWS is not None and len(usable) > MAX_LABEL_ROWS:
        groups = []
        labels = sorted(usable['binary_label'].unique())
        base_n = max(1, MAX_LABEL_ROWS // max(len(labels), 1))
        for label, group in usable.groupby('binary_label'):
            groups.append(group.sample(n=min(len(group), base_n), random_state=42))
        sampled = pd.concat(groups, ignore_index=True)
        if len(sampled) < MAX_LABEL_ROWS:
            remaining = usable.drop(index=sampled.index, errors='ignore')
            if not remaining.empty:
                sampled = pd.concat([
                    sampled,
                    remaining.sample(n=min(MAX_LABEL_ROWS - len(sampled), len(remaining)), random_state=43)
                ], ignore_index=True)
        usable = sampled.sample(frac=1.0, random_state=44).reset_index(drop=True)

    PUBLIC_TRAINING_CSV.parent.mkdir(parents=True, exist_ok=True)
    usable.to_csv(PUBLIC_TRAINING_CSV, index=False)
    CURATED_CSV = PUBLIC_TRAINING_CSV
    LABEL_COL = 'binary_label'
    CURATED_SOURCE = None
    EPOCH_OFFSET = None
else:
    CURATED_CSV = CURATED_INPUT_CSV
    LABEL_COL = 'label'
    CURATED_SOURCE = None
    # Change this if your curated labels are full BJD instead of TESS BTJD.
    EPOCH_OFFSET = None

assert_exists(CURATED_CSV, 'label/metadata CSV for feature building')
labels_df = pd.read_csv(CURATED_CSV)
print('Feature-build label table:', labels_df.shape)
display(labels_df.head())
print('Label counts:')
print(labels_df[LABEL_COL].value_counts(dropna=False))


## 6. Optional Synthetic Smoke Test

This is off by default. It is not part of the real-data result.


In [ ]:
if RUN_SYNTHETIC_SMOKE:
    run_command([
        sys.executable, 'scripts/run_parts_9_10_synthetic_batch.py',
        '--output-dir', str(WORK / 'smoke_outputs'),
        '--n-periods', '500',
    ], cwd=REPO_DIR)
else:
    print('Skipping synthetic smoke test. Current run is real-data mode:', REAL_DATA_MODE)


## 7. Build Labeled Candidate Feature Catalog From Real FITS

This downloads missing real TESS FITS from MAST, preprocesses light curves, estimates transit/event features, and writes a labeled feature catalog. A compact single-line progress bar is updated during the run, and checkpoint CSVs are written periodically.


In [ ]:
%%time
feature_lc_dir = LIGHTCURVE_DIR if LIGHTCURVE_DIR.exists() and count_fits(LIGHTCURVE_DIR) else LIGHTCURVE_DOWNLOAD_DIR
feature_lc_dir.mkdir(parents=True, exist_ok=True)
print('Light-curve cache/download directory:', feature_lc_dir)
print('Existing FITS files before feature build:', count_fits(feature_lc_dir))

build_cmd = [
    sys.executable, 'scripts/build_labeled_candidate_catalog.py',
    str(CURATED_CSV),
    '--label-col', LABEL_COL,
    '--lightcurve-dir', str(feature_lc_dir),
    '--output-dir', str(OUT_LABELS),
    '--n-periods', str(N_PERIODS_FEATURES),
    '--n-workers', str(N_WORKERS_FEATURES),
    '--n-bootstrap', str(N_BOOTSTRAP_FEATURES),
    '--checkpoint-every', str(FEATURE_CHECKPOINT_EVERY),
    '--progress-every', '1',
    '--progress-style', 'bar',
    '--max-lightcurves-per-target', '1',
]
if CURATED_SOURCE is not None:
    build_cmd.extend(['--source', CURATED_SOURCE])
if USE_MAST_DOWNLOADS:
    build_cmd.append('--download-missing')
if EPOCH_OFFSET is not None:
    build_cmd.extend(['--epoch-offset', str(EPOCH_OFFSET)])
if NO_VARIANTS:
    build_cmd.append('--no-variants')

run_command(build_cmd, cwd=REPO_DIR)
print('Existing FITS files after feature build:', count_fits(feature_lc_dir))


## 8. Inspect Extracted Features

Do not train if this is empty or one-class only. Check manifest statuses first.


In [ ]:
features_path = OUT_LABELS / 'labeled_candidate_features.csv'
manifest_path = OUT_LABELS / 'labeled_candidate_manifest.csv'
summary_path = OUT_LABELS / 'labeled_candidate_summary.json'

assert_exists(features_path, 'labeled feature catalog')
assert_exists(manifest_path, 'feature-build manifest')

features = pd.read_csv(features_path)
manifest = pd.read_csv(manifest_path)
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

print(json.dumps(summary, indent=2))
print('Feature shape:', features.shape)
display(features.head())
print('\nFeature labels:')
print(features['label'].value_counts(dropna=False) if 'label' in features else 'No label column')
print('\nManifest statuses:')
print(manifest['status'].value_counts(dropna=False))
display(manifest.head())

if features.empty:
    raise RuntimeError('No labeled feature rows were built. Check MAST download errors and manifest statuses.')
if features['label'].nunique(dropna=True) < 2:
    raise RuntimeError('Only one label class survived feature building. Increase PUBLIC_TOI_TOP/MAX_LABEL_ROWS or inspect label sampling.')


## 9. Train Tabular AI Classifier

This trains the supervised classifier used by the science pipeline. In public TOI mode this is a real-data binary training run: planet-like vs false-positive/other.


In [ ]:
%%time
run_command([
    sys.executable, 'scripts/train_ai_classifier_from_catalog.py',
    str(features_path),
    '--label-col', 'label',
    '--output-dir', str(OUT_AI),
    '--model-type', 'random_forest',
], cwd=REPO_DIR)


## 10. Inspect Model Metrics


In [ ]:
metrics_path = OUT_AI / 'part6_ai_classifier_metrics.json'
importance_path = OUT_AI / 'part6_ai_classifier_feature_importance.csv'
assert_exists(metrics_path, 'model metrics')
metrics = json.loads(metrics_path.read_text())

print('Classes:', metrics.get('class_names'))
print('Training metadata:', metrics.get('training_metadata'))
print('\nHoldout/test metrics:')
for key, value in metrics.get('test_metrics', {}).items():
    if key != 'classification_report':
        print(f'{key}: {value}')
print('\nClassification report:')
print(json.dumps(metrics.get('test_metrics', {}).get('classification_report', {}), indent=2)[:4000])

if importance_path.exists():
    print('\nTop feature importances:')
    display(pd.read_csv(importance_path).head(20))


## 11. Optional CNN Training

Leave this off for the first real run. It is separate from the tabular classifier above.


In [ ]:
if RUN_OPTIONAL_CNN:
    run_command([
        sys.executable, 'scripts/build_public_cnn_examples.py',
        str(PUBLIC_NORMALIZED_TOI_CSV),
        '--output-dir', str(PUBLIC_DATA_DIR / 'cnn_examples'),
        '--lightcurve-dir', str(PUBLIC_DATA_DIR / 'cnn_lightcurves'),
        '--download-missing',
        '--max-rows', str(CNN_MAX_ROWS),
        '--n-workers', '8',
    ], cwd=REPO_DIR)
    run_command([
        sys.executable, 'scripts/train_cnn_classifier.py',
        str(PUBLIC_DATA_DIR / 'cnn_examples'),
        '--output-dir', str(OUT_CNN),
        '--epochs', str(CNN_EPOCHS),
        '--batch-size', str(CNN_BATCH_SIZE),
        '--device', 'cuda' if GPU_AVAILABLE else 'cpu',
    ], cwd=REPO_DIR)
else:
    print('Skipping optional CNN training.')


## 12. Run Science Batch On Real FITS

For the default public run, this reuses the real FITS downloaded during feature building. For a proper blind science run, point `SCIENCE_FITS_DIR` to a separate sector/FITS dataset and rerun this section.


In [ ]:
%%time
ai_model_path = OUT_AI / 'part6_ai_classifier.joblib'
assert_exists(ai_model_path, 'trained tabular AI model')

science_fits_dir = feature_lc_dir
n_fits = count_fits(science_fits_dir)
print('Science FITS directory:', science_fits_dir)
print('Science FITS count:', n_fits)
if n_fits == 0:
    raise RuntimeError('No FITS files available for science run. Feature-building did not download or find light curves.')

science_cmd = [
    sys.executable, 'scripts/run_parts_9_10_fits_directory.py',
    str(science_fits_dir),
    '--output-dir', str(OUT_SCIENCE),
    '--ai-model', str(ai_model_path),
    '--n-workers', str(N_WORKERS_SCIENCE),
    '--n-periods', str(N_PERIODS_SCIENCE),
    '--timeout-seconds', str(TIMEOUT_SECONDS),
]
if RUN_OPTIONAL_CNN and (OUT_CNN / 'cnn_model.pt').exists():
    science_cmd.extend(['--cnn-model', str(OUT_CNN)])
if NO_VARIANTS:
    science_cmd.append('--no-variants')

run_command(science_cmd, cwd=REPO_DIR)


## 13. Inspect Science Outputs


In [ ]:
catalog_path = OUT_SCIENCE / 'batch_final_candidate_catalog.csv'
summary_path = OUT_SCIENCE / 'batch_target_summary.csv'
failure_path = OUT_SCIENCE / 'batch_failure_log.csv'

catalog = pd.read_csv(catalog_path) if catalog_path.exists() else pd.DataFrame()
target_summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
failures = pd.read_csv(failure_path) if failure_path.exists() else pd.DataFrame()

print('Final candidate catalog:', catalog.shape)
display(catalog.head(10))
if 'final_predicted_class' in catalog.columns:
    print('\nFinal predicted classes:')
    print(catalog['final_predicted_class'].value_counts(dropna=False))

print('\nTarget summary:', target_summary.shape)
if 'status' in target_summary.columns:
    print(target_summary['status'].value_counts(dropna=False))
display(target_summary.head(10))

print('\nFailures:', failures.shape)
display(failures.head(10))


## 14. Optional Validation

Run this only if you have labels joined onto the final science candidate catalog. Public TOI training labels are not automatically the same as blind science candidate labels.


In [ ]:
validation_report = OUT_VALIDATION / 'validation_report.json'
if LABELED_SCIENCE_CATALOG is not None and Path(LABELED_SCIENCE_CATALOG).exists():
    run_command([
        sys.executable, 'scripts/validate_candidate_catalog.py',
        '--catalog', str(LABELED_SCIENCE_CATALOG),
        '--label-col', 'label',
        '--pred-col', 'final_predicted_class',
        '--out-dir', str(OUT_VALIDATION),
    ], cwd=REPO_DIR)
else:
    print('Skipping validation. Set LABELED_SCIENCE_CATALOG to a labeled final candidate catalog to enable it.')
    validation_report = None


## 15. Generate Final Assets And Report Draft


In [ ]:
assert_exists(catalog_path, 'science final candidate catalog')
asset_cmd = [
    sys.executable, 'scripts/generate_final_submission_assets.py',
    str(catalog_path),
    '--output-dir', str(OUT_SCIENCE / 'submission_assets'),
]
if validation_report is not None and Path(validation_report).exists():
    asset_cmd.extend(['--validation-report', str(validation_report)])
run_command(asset_cmd, cwd=REPO_DIR)

print('Submission assets:')
for path in sorted((OUT_SCIENCE / 'submission_assets').glob('*')):
    print(path)


## 16. Package Outputs


In [ ]:
import zipfile

zip_out = WORK / 'exoplanet_pipeline_outputs.zip'
if zip_out.exists():
    zip_out.unlink()

folders = [OUT_LABELS, OUT_AI, OUT_SCIENCE]
if OUT_VALIDATION.exists():
    folders.append(OUT_VALIDATION)
if OUT_CNN.exists():
    folders.append(OUT_CNN)

n_files = 0
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for folder in folders:
        if not folder.exists():
            continue
        for item in folder.rglob('*'):
            if item.is_file():
                zipf.write(item, arcname=item.relative_to(WORK))
                n_files += 1

print(f'Archived {n_files} files into {zip_out}')
print(f'Zip size: {zip_out.stat().st_size / (1024 * 1024):.2f} MB')


## What This Run Does And Does Not Prove

This notebook uses real public TOI metadata and real TESS FITS. It is suitable for proving that the pipeline can ingest real data, train a real classifier, estimate period/depth/duration/SNR, and produce final artifacts.

The public TOI labels are broad binary labels. For the full project objective's multi-class taxonomy (`transits`, `eclipses`, `blends`, `starspots/other`), use the organizer-provided curated labels in `REAL_DATA_MODE='curated'` once available.
